In [2]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch_geometric.utils import degree
import networkx as nx
import warnings

warnings.filterwarnings("ignore")


In [3]:
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(exist_ok=True)

## Synthetic Fraud Data Cleaning
- Missing Value Check
- Edge Validity Check
- Outlier Handling

In [4]:
print("Loading Synthetic Fraud Dataset...")
data = torch.load(DATA_PROCESSED / "synthetic_fraud.pt", weights_only=False)

print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)


Loading Synthetic Fraud Dataset...
Nodes: 5000
Edges: 40900


In [5]:
node_nan = torch.isnan(data.x).sum()
edge_nan = torch.isnan(data.edge_attr).sum()

print("Node feature NaNs:", node_nan.item())
print("Edge feature NaNs:", edge_nan.item())


Node feature NaNs: 0
Edge feature NaNs: 0


In [6]:
src, dst = data.edge_index

invalid_edges = ((src >= data.num_nodes) | (dst >= data.num_nodes)).sum()
print("Invalid edges:", invalid_edges.item())

self_loops = (src == dst).sum()
print("Self-loops:", self_loops.item())

# Remove invalid edges if any
valid_mask = (src < data.num_nodes) & (dst < data.num_nodes)
data.edge_index = data.edge_index[:, valid_mask]
data.edge_attr = data.edge_attr[valid_mask]


Invalid edges: 0
Self-loops: 5


In [7]:
edge_df = pd.DataFrame(data.edge_attr.numpy())
lower = edge_df.quantile(0.01)
upper = edge_df.quantile(0.99)

edge_df = edge_df.clip(lower=lower, upper=upper, axis=1)

data.edge_attr = torch.tensor(edge_df.values, dtype=torch.float32)


In [8]:
torch.save(data, DATA_PROCESSED / "synthetic_fraud_clean.pt")
print("Synthetic dataset cleaned and saved.")


Synthetic dataset cleaned and saved.


## Online Payments Cleaning
- Missing Value Check
- Data Type Standardization
- Outlier Handling
- Transaction Type Encoding

In [25]:
print("Loading Online Payments Dataset...")
df = pd.read_csv(DATA_RAW / "online_payments.csv")

print("Initial Shape:", df.shape)
df.head()


Loading Online Payments Dataset...
Initial Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [26]:
missing = df.isnull().sum()

missing_df = pd.DataFrame({
    "Column": missing.index,
    "Missing_Count": missing.values,
    "Missing_Percentage": (missing.values / len(df)) * 100
})

missing_df


,Column,Missing_Count,Missing_Percentage
0,step,0,0.0
1,type,0,0.0
2,amount,0,0.0
3,nameOrig,0,0.0
4,oldbalanceOrg,0,0.0
5,newbalanceOrig,0,0.0
6,nameDest,0,0.0
7,oldbalanceDest,0,0.0
8,newbalanceDest,0,0.0
9,isFraud,0,0.0


In [27]:
# Type conversion
df['type'] = df['type'].astype('category')
df['isFraud'] = df['isFraud'].astype(int)
df['isFlaggedFraud'] = df['isFlaggedFraud'].astype(int)

print("Type conversion completed successfully.")

Type conversion completed successfully.


In [28]:
# Log Transformation-
print("Log Transformation on Amount")
print("Before (amount stats):")
print(df['amount'].describe())
print()

df['log_amount'] = np.log1p(df['amount'])

print("After (log_amount stats):")
print(df['log_amount'].describe())
print()

# Balance Difference (Before Clipping)
df['orig_balance_diff'] = (
    df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
)

print("Balance Difference BEFORE Clipping:")
print(df['orig_balance_diff'].describe())
print()

lower = df['orig_balance_diff'].quantile(0.01)
upper = df['orig_balance_diff'].quantile(0.99)

print(f"Clipping range: [{lower:.2f}, {upper:.2f}]")

# Apply clipping
df['orig_balance_diff'] = df['orig_balance_diff'].clip(lower, upper)

print("\nBalance Difference AFTER Clipping:")
print(df['orig_balance_diff'].describe())
print()

print("Outlier handling completed successfully.")


Log Transformation on Amount
Before (amount stats):
count    6.362620e+06
mean     1.798619e+05
std      6.038582e+05
min      0.000000e+00
25%      1.338957e+04
50%      7.487194e+04
75%      2.087215e+05
max      9.244552e+07
Name: amount, dtype: float64

After (log_amount stats):
count    6.362620e+06
mean     1.084087e+01
std      1.814509e+00
min      0.000000e+00
25%      9.502306e+00
50%      1.122355e+01
75%      1.224876e+01
max      1.834213e+01
Name: log_amount, dtype: float64

Balance Difference BEFORE Clipping:
count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: orig_balance_diff, dtype: float64

Clipping range: [-1559494.91, 0.00]

Balance Difference AFTER Clipping:
count    6.362620e+06
mean    -1.788093e+05
std      2.699590e+05
min     -1.559495e+06
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.164153e-10
Name:

In [29]:
print("Applying One-Hot Encoding on 'type' column...")

if 'type' in df.columns:
    before_shape = df.shape
    
    df = pd.get_dummies(df, columns=['type'], drop_first=True)
    
    after_shape = df.shape
    
    print("Encoding completed successfully.")
    print(f"Shape before encoding: {before_shape}")
    print(f"Shape after encoding:  {after_shape}")
else:
    print("'type' column not found.")
    print("It has likely already been encoded earlier.")




Applying One-Hot Encoding on 'type' column...
Encoding completed successfully.
Shape before encoding: (6362620, 13)
Shape after encoding:  (6362620, 16)


In [30]:
df.to_csv(DATA_PROCESSED / "online_payments_clean.csv", index=False)
print("Online payments dataset cleaned and saved.")


Online payments dataset cleaned and saved.


## Elliptic Data Cleaning
- Missing Value check
- Handling Unlabelled data
- Feature Normalization
- Edge Validation

In [37]:
print("Loading Elliptic Dataset...\n")

# Load Files
df_feat = pd.read_csv(DATA_RAW / "elliptic_txs_features.csv", header=None)
df_class = pd.read_csv(DATA_RAW / "elliptic_txs_classes.csv")
df_edge = pd.read_csv(DATA_RAW / "elliptic_txs_edgelist.csv")

print("Files loaded successfully.\n")

# Basic Shape Information
print("Features File Shape:", df_feat.shape)
print("Classes File Shape: ", df_class.shape)
print("Edges File Shape:   ", df_edge.shape)
print()


Loading Elliptic Dataset...

Files loaded successfully.

Features File Shape: (203769, 167)
Classes File Shape:  (203769, 2)
Edges File Shape:    (234355, 2)



In [39]:
missing_feat = df_feat.isnull().sum().sum()
print("Total missing values in df_feat:", missing_feat)

missing_class = df_class.isnull().sum().sum()
print("Total missing values in df_class:", missing_class)

missing_edge = df_edge.isnull().sum().sum()
print("Total missing values in df_edge:", missing_edge)


Total missing values in df_feat: 0
Total missing values in df_class: 0
Total missing values in df_edge: 0


In [33]:
df_class['is_labeled'] = df_class['class'] != 'unknown'
df_class['class_clean'] = df_class['class'].replace({'unknown': -1}).astype(int)

print(df_class['class_clean'].value_counts())


class_clean
-1    157205
 2     42019
 1      4545
Name: count, dtype: int64


In [34]:
features = df_feat.iloc[:, 2:].values
features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-6)

df_feat.iloc[:, 2:] = features

print("Normalization completed successfully.")
print("Updated dataframe shape:", df_feat.shape)

Normalization completed successfully.
Updated dataframe shape: (203769, 167)


In [35]:
valid_ids = set(df_feat.iloc[:, 0])

df_edge = df_edge[
    df_edge.iloc[:,0].isin(valid_ids) &
    df_edge.iloc[:,1].isin(valid_ids)
]

print("Valid edges:", len(df_edge))


Valid edges: 234355


In [36]:
df_feat.to_csv(DATA_PROCESSED / "elliptic_features_clean.csv", index=False)
df_class.to_csv(DATA_PROCESSED / "elliptic_classes_clean.csv", index=False)
df_edge.to_csv(DATA_PROCESSED / "elliptic_edges_clean.csv", index=False)

print("Elliptic dataset cleaned and saved.")


Elliptic dataset cleaned and saved.
